# Experiment – Reliable Healthcare Advisor Agent with Tool Integration

**Problem Statement:** Build a **Medication Safety & Adherence Advisor** that helps a user understand prescribed medicines, screen a medicine list for a small set of duplicate-ingredient risks, and provide controlled missed-dose education. The agent uses structured tool calls, validation, authorization, retries, approved fallback, logging, and reliability evaluation without diagnosing or prescribing.

**Required LLM:** Gemini 3.5 Flash (`gemini-3.5-flash`)  
**Request timeout:** 12.5 seconds  
**API key:** entered at runtime; never hardcoded.

The supplied lab sequence is followed exactly, then the six optimizations, then exactly five reliability checks: **Output Accuracy, Tool-Calling Reliability, Recovery/Fault Tolerance, Cost Reliability, and Observability**.

## Case Study Adaptation

The healthcare domain remains the same as the supplied experiment, but the problem is different from BMI/weight calculation.

The agent provides medication-information lookup, duplicate-ingredient screening, and missed-dose educational guidance using controlled local knowledge/rules. It is educational decision support only and does not diagnose, prescribe, or change medication.

## Step 1: Import Required Libraries

In [ ]:
import json
import time
import random
import csv
import re
import os
from datetime import datetime
from typing import Dict, Any, List, Optional

## Step 2: Define the Standard Tool Contract

In [ ]:
TOOL_CONTRACT = {
    "tool_name": "",
    "parameters": {},
    "status": "",
    "result": None,
    "error": None
}

def contract(tool_name="", parameters=None, status="", result=None, error=None):
    return {
        "tool_name": tool_name,
        "parameters": parameters or {},
        "status": status,
        "result": result,
        "error": error
    }

## Step 3: Create Healthcare Tools

In [ ]:
MEDICATION_KB = {
    "warfarin": {"class": "anticoagulant",
                 "education": "Warfarin reduces the blood's ability to form clots. It requires individualized monitoring and clinician guidance."},
    "metformin": {"class": "antidiabetic medicine",
                  "education": "Metformin is commonly used to help control blood glucose in type 2 diabetes."},
    "amoxicillin": {"class": "antibiotic",
                    "education": "Amoxicillin is a penicillin-type antibiotic used for certain bacterial infections."},
    "acetaminophen": {"class": "pain reliever/fever reducer",
                      "education": "Acetaminophen is used to reduce pain or fever. Total intake from all products should be checked carefully."}
}

MISSED_DOSE_KB = {
    "warfarin": "For a missed warfarin dose, follow the medication label or prescriber/pharmacist instructions; do not make an unsupervised dose change.",
    "metformin": "For a missed metformin dose, follow the medication label or prescriber/pharmacist instructions rather than doubling a dose.",
    "amoxicillin": "For a missed amoxicillin dose, follow the medication label or prescriber/pharmacist instructions rather than doubling a dose.",
    "acetaminophen": "For a missed acetaminophen dose, follow the product label or clinician/pharmacist instructions and avoid exceeding labeled dosing limits."
}

DUPLICATE_INGREDIENT_PAIRS = {
    tuple(sorted(("acetaminophen", "paracetamol"))):
        "Potential duplicate ingredient: acetaminophen and paracetamol refer to the same active ingredient."
}

def medication_knowledge_search(query: str, max_results: int = 3) -> Dict[str, Any]:
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query must be a non-empty string")
    if not isinstance(max_results, int) or max_results < 1 or max_results > 5:
        raise ValueError("max_results must be an integer from 1 to 5")
    q = query.lower()
    matches = [{"medication": name, **info} for name, info in MEDICATION_KB.items() if name in q]
    return {"source": "Controlled Medication KB", "matches": matches[:max_results]}

def interaction_screen(medicines: List[str]) -> Dict[str, Any]:
    if not isinstance(medicines, list) or not medicines:
        raise ValueError("medicines must be a non-empty list")
    normalized = [str(m).strip().lower() for m in medicines if str(m).strip()]
    findings = []
    for i in range(len(normalized)):
        for j in range(i + 1, len(normalized)):
            pair = tuple(sorted((normalized[i], normalized[j])))
            if pair in DUPLICATE_INGREDIENT_PAIRS:
                findings.append(DUPLICATE_INGREDIENT_PAIRS[pair])
    if not findings:
        findings = ["No duplicate active-ingredient pair was detected in the controlled rule set."]
    return {"source": "Controlled Interaction Rules", "medicines": normalized, "findings": findings}

def missed_dose_guidance(medication: str) -> Dict[str, Any]:
    if not isinstance(medication, str) or not medication.strip():
        raise ValueError("medication must be a non-empty string")
    name = medication.strip().lower()
    guidance = MISSED_DOSE_KB.get(name)
    if guidance is None:
        return {"source": "Controlled Missed-Dose KB", "status": "NO_RESULT",
                "guidance": "No controlled missed-dose guidance is available. Follow the product label or contact a pharmacist/clinician."}
    return {"source": "Controlled Missed-Dose KB", "status": "OK", "guidance": guidance}

TOOL_REGISTRY = {
    "medication_knowledge_search": medication_knowledge_search,
    "interaction_screen": interaction_screen,
    "missed_dose_guidance": missed_dose_guidance
}

## Step 4: Create Tool Registry

In [ ]:
print("Registered tools:", list(TOOL_REGISTRY.keys()))

Registered tools: ['medication_knowledge_search', 'interaction_screen', 'missed_dose_guidance']


## Step 5: Define Tool Descriptions

In [ ]:
TOOL_DESCRIPTIONS = {
    "medication_knowledge_search": {
        "description": "Look up educational information about a known medicine.",
        "keywords": ["what is", "medication", "medicine", "drug", "warfarin", "metformin", "amoxicillin", "acetaminophen"]
    },
    "interaction_screen": {
        "description": "Check a medicine list against approved duplicate-ingredient rules.",
        "keywords": ["duplicate", "same ingredient", "interaction", "together", "combine", "medicines", "paracetamol", "acetaminophen"]
    },
    "missed_dose_guidance": {
        "description": "Return controlled educational guidance for a missed dose.",
        "keywords": ["missed dose", "forgot dose", "forgot to take", "skip dose"]
    }
}

## Step 6: Implement Tool Selection

In [ ]:
def select_tool(query: str) -> str:
    q = query.lower().strip()
    if any(k in q for k in TOOL_DESCRIPTIONS["missed_dose_guidance"]["keywords"]):
        return "missed_dose_guidance"
    if any(k in q for k in TOOL_DESCRIPTIONS["interaction_screen"]["keywords"]):
        return "interaction_screen"
    if any(k in q for k in TOOL_DESCRIPTIONS["medication_knowledge_search"]["keywords"]):
        return "medication_knowledge_search"
    return "NO_TOOL"

selection_tests = [
    ("What is warfarin?", "medication_knowledge_search"),
    ("Can acetaminophen and paracetamol be duplicated?", "interaction_screen"),
    ("What should I do after a missed dose of metformin?", "missed_dose_guidance")
]
for q, expected in selection_tests:
    print(q, "->", select_tool(q), "| expected:", expected)

What is warfarin? -> medication_knowledge_search | expected: medication_knowledge_search
Can acetaminophen and paracetamol be duplicated? -> interaction_screen | expected: interaction_screen
What should I do after a missed dose of metformin? -> missed_dose_guidance | expected: missed_dose_guidance


In [ ]:
# Dynamic key input: never hardcode the API key.
import getpass

GEMINI_MODEL = "gemini-3.5-flash"
REQUEST_TIMEOUT_SECONDS = 12.5
MAX_LLM_OUTPUT_TOKENS = 500

GEMINI_API_KEY = getpass.getpass("Enter your Gemini API key (input hidden): ")

try:
    from google import genai
    from google.genai import types
    GEMINI_CLIENT = genai.Client(
        api_key=GEMINI_API_KEY,
        http_options=types.HttpOptions(
            timeout=int(REQUEST_TIMEOUT_SECONDS * 1000),
            retry_options=types.HttpRetryOptions(attempts=1)
        )
    )
    GEMINI_STATUS = "READY"
except Exception as e:
    GEMINI_CLIENT = None
    GEMINI_STATUS = f"SDK/API setup unavailable: {e}"

print("Model:", GEMINI_MODEL)
print("Timeout:", REQUEST_TIMEOUT_SECONDS, "seconds")
print("Gemini status:", GEMINI_STATUS)

Enter your Gemini API key (input hidden): ··········
Model: gemini-3.5-flash
Timeout: 12.5 seconds
Gemini status: READY


In [ ]:
def gemini_generate(prompt: str) -> Dict[str, Any]:
    if GEMINI_CLIENT is None:
        return {"ok": False, "error": GEMINI_STATUS, "output": ""}
    start = time.perf_counter()
    try:
        response = GEMINI_CLIENT.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=(
                    "You are a healthcare advisor for educational decision support. "
                    "Do not diagnose, prescribe, or invent medical facts. "
                    "Use only the supplied tool result as evidence. "
                    "State when the evidence is unavailable."
                ),
                max_output_tokens=MAX_LLM_OUTPUT_TOKENS
            )
        )
        elapsed = time.perf_counter() - start
        usage = getattr(response, "usage_metadata", None)
        input_tokens = getattr(usage, "prompt_token_count", 0) if usage else 0
        output_tokens = getattr(usage, "candidates_token_count", 0) if usage else 0
        text = getattr(response, "text", "") or ""
        if elapsed > REQUEST_TIMEOUT_SECONDS:
            return {"ok": False, "error": "REQUEST_TIMEOUT_12_5S", "output": "",
                    "elapsed": elapsed, "input_tokens": input_tokens, "output_tokens": output_tokens}
        return {"ok": True, "output": text, "elapsed": elapsed,
                "input_tokens": input_tokens, "output_tokens": output_tokens}
    except Exception as e:
        return {"ok": False, "error": type(e).__name__, "output": "",
                "elapsed": time.perf_counter() - start, "input_tokens": 0, "output_tokens": 0}



## Step 7: Extract Tool Parameters

In [ ]:
def extract_parameters(query: str, tool_name: str) -> Dict[str, Any]:
    q = query.strip()
    if tool_name == "medication_knowledge_search":
        return {"query": q, "max_results": 3}
    if tool_name == "missed_dose_guidance":
        m = re.search(r"(?:missed dose of|forgot.*?dose of)\s+([A-Za-z]+)", q, re.I)
        if m:
            return {"medication": m.group(1).lower()}
        for med in MEDICATION_KB:
            if med in q.lower():
                return {"medication": med}
        return {}
    if tool_name == "interaction_screen":
        found = [m for m in ["acetaminophen", "paracetamol", "warfarin", "metformin", "amoxicillin"] if m in q.lower()]
        return {"medicines": found}
    return {}

## Step 8: Create Structured Tool Calls

In [ ]:
def create_tool_call(tool_name: str, parameters: Dict[str, Any]) -> Dict[str, Any]:
    return {"status": "READY", "tool_name": tool_name, "parameters": parameters, "error": None}

demo_call = create_tool_call("missed_dose_guidance", {"medication": "metformin"})
print(json.dumps(demo_call, indent=2))

{
  "status": "READY",
  "tool_name": "missed_dose_guidance",
  "parameters": {
    "medication": "metformin"
  },
  "error": null
}


## Step 9: Define Tool Schemas

In [ ]:
TOOL_SCHEMAS = {
    "medication_knowledge_search": {"query": str, "max_results": int},
    "interaction_screen": {"medicines": list},
    "missed_dose_guidance": {"medication": str}
}
print(json.dumps({k: {p: t.__name__ for p, t in v.items()} for k, v in TOOL_SCHEMAS.items()}, indent=2))

{
  "medication_knowledge_search": {
    "query": "str",
    "max_results": "int"
  },
  "interaction_screen": {
    "medicines": "list"
  },
  "missed_dose_guidance": {
    "medication": "str"
  }
}


## Step 10: Validate Structured Tool Calls

In [ ]:
def validate_tool_call(tool_call: Dict[str, Any]) -> Optional[str]:
    if not isinstance(tool_call, dict):
        return "call must be a dictionary"
    tool_name = tool_call.get("tool_name")
    if tool_name not in TOOL_SCHEMAS:
        return "unknown tool"
    params = tool_call.get("parameters")
    if not isinstance(params, dict):
        return "parameters must be a dictionary"
    for required, expected_type in TOOL_SCHEMAS[tool_name].items():
        if required not in params:
            return f"missing parameter: {required}"
        if not isinstance(params[required], expected_type):
            return f"invalid type for {required}: expected {expected_type.__name__}"
    if tool_name == "interaction_screen" and not params["medicines"]:
        return "medicines must not be empty"
    if tool_name == "medication_knowledge_search" and not params["query"].strip():
        return "query must not be empty"
    return None

print("Valid:", validate_tool_call(demo_call))
print("Invalid:", validate_tool_call(create_tool_call("missed_dose_guidance", {"medication": 123})))

Valid: None
Invalid: invalid type for medication: expected str


## Step 12: Implement Tool Authorization

In [ ]:
ALLOWED_TOOLS = {
    "medication_knowledge_search",
    "interaction_screen",
    "missed_dose_guidance"
}

def authorize_tool(tool_name: str) -> bool:
    return tool_name in ALLOWED_TOOLS

print("Authorized known tool:", authorize_tool("missed_dose_guidance"))
print("Unauthorized tool:", authorize_tool("delete_medical_database"))

Authorized known tool: True
Unauthorized tool: False


## Step 14: Implement Error Classification

In [ ]:
ERROR_TYPES = {
    "INPUT_ERROR": (ValueError, TypeError),
    "AUTHORIZATION_ERROR": (PermissionError,),
    "TRANSIENT_ERROR": (TimeoutError, ConnectionError),
    "SERVICE_ERROR": (RuntimeError,)
}

def classify_error(exc: Exception) -> str:
    for name, types in ERROR_TYPES.items():
        if isinstance(exc, types):
            return name
    return "UNKNOWN_ERROR"

examples = [ValueError("bad input"), PermissionError("blocked"), TimeoutError("temporary"),
            RuntimeError("service"), Exception("unknown")]
for exc in examples:
    print(type(exc).__name__, "->", classify_error(exc))


ValueError -> INPUT_ERROR
PermissionError -> AUTHORIZATION_ERROR
TimeoutError -> TRANSIENT_ERROR
RuntimeError -> SERVICE_ERROR
Exception -> UNKNOWN_ERROR


## Step 11: Implement Tool Execution Engine

In [ ]:
def execute_tool(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    validation_error = validate_tool_call(tool_call)
    if validation_error:
        return contract(tool_call.get("tool_name", ""), tool_call.get("parameters", ""),
                        "VALIDATION_FAILED", None,
                        {"category": "INPUT_ERROR", "message": validation_error})
    tool_name = tool_call["tool_name"]
    try:
        result = TOOL_REGISTRY[tool_name](**tool_call["parameters"])
        return contract(tool_name, tool_call["parameters"], "SUCCESS", result, None)
    except Exception as exc:
        return contract(tool_name, tool_call["parameters"], "ERROR", None,
                        {"category": classify_error(exc), "message": str(exc)})



## Step 13: Implement Retry Mechanism

In [ ]:
MAX_RETRIES = 2
TOTAL_ATTEMPTS = MAX_RETRIES + 1

def execute_with_retry(tool_call: Dict[str, Any], failure_injector=None) -> Dict[str, Any]:
    attempts = 0
    last = None
    while attempts < TOTAL_ATTEMPTS:
        attempts += 1
        try:
            if failure_injector is not None:
                failure_injector(attempts)
            last = execute_tool(tool_call)
            if last["status"] == "SUCCESS":
                last["attempts"] = attempts
                return last
            category = (last.get("error") or {}).get("category")
            if category not in {"TRANSIENT_ERROR", "SERVICE_ERROR"}:
                last["attempts"] = attempts
                return last
        except Exception as exc:
            last = contract(tool_call.get("tool_name", ""), tool_call.get("parameters", ""),
                            "ERROR", None,
                            {"category": classify_error(exc), "message": str(exc)})
            if last["error"]["category"] not in {"TRANSIENT_ERROR", "SERVICE_ERROR"}:
                last["attempts"] = attempts
                return last
        time.sleep(0.02)
    last["attempts"] = attempts
    return last

## Step 15: Implement Fallback Mechanism

In [ ]:
FALLBACK_TOOLS = {
    "interaction_screen": "medication_knowledge_search",
    "missed_dose_guidance": "medication_knowledge_search"
}

def fallback_call(primary_tool: str, parameters: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    fallback = FALLBACK_TOOLS.get(primary_tool)
    if fallback is None:
        return None
    if fallback == "medication_knowledge_search":
        query_text = " ".join(parameters.get("medicines", [])) or parameters.get("medication", "")
        fallback_params = {"query": query_text, "max_results": 3}
    else:
        fallback_params = parameters
    return execute_tool(create_tool_call(fallback, fallback_params))

## Step 16: Implement Safe Failure

In [ ]:
def safe_failure(message="The request could not be answered reliably from the approved healthcare tools."):
    return {"status": "SAFE_FAILURE", "response": message, "source": None}

def safe_response(tool_result: Dict[str, Any]) -> Dict[str, Any]:
    if tool_result["status"] == "SUCCESS":
        data = tool_result["result"]
        return {"status": "SAFE_SUCCESS", "response": json.dumps(data),
                "source": data.get("source") if isinstance(data, dict) else None}
    return safe_failure()

## Step 17: Implement Interaction Logging

In [ ]:
# Reset old interaction log after changing the log format
if os.path.exists("healthcare_advisor_interaction_log.csv"):
    os.remove("healthcare_advisor_interaction_log.csv")
    print("Old interaction log removed. It will be recreated with the new 4-column format.")


In [ ]:
# Step 17: Simple Interaction Logging

LOG_FILE = "healthcare_advisor_interaction_log.csv"
LOG_FIELDS = ["timestamp", "query", "tool", "response"]

def log_interaction(query, tool, response):
    exists = os.path.exists(LOG_FILE)
    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if not exists:
            writer.writeheader()
        writer.writerow({
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "query": query,
            "tool": tool,
            "response": response
        })


## Step 21: Execute the Complete Agent

In [ ]:
# Step 21: Complete Agent

def run_agent(user_query: str, force_tool_failure: bool = False) -> Dict[str, Any]:
    if not isinstance(user_query, str) or not user_query.strip():
        result = safe_failure("Input error: please provide a healthcare question.")
        log_interaction(user_query, "NO_TOOL", result["response"])
        return result
    tool_name = select_tool(user_query)
    if tool_name == "NO_TOOL":
        result = safe_failure("No approved healthcare tool matches this request.")
        log_interaction(user_query, "NO_TOOL", result["response"])
        return result
    params = extract_parameters(user_query, tool_name)
    call = create_tool_call(tool_name, params)
    schema_error = validate_tool_call(call)
    if schema_error:
        result = safe_failure(f"Structured tool call rejected: {schema_error}.")
        log_interaction(user_query, tool_name, result["response"])
        return result
    if not authorize_tool(tool_name):
        result = safe_failure("Tool authorization rejected.")
        log_interaction(user_query, tool_name, result["response"])
        return result
    def injected_failure(attempt):
        if force_tool_failure:
            raise TimeoutError("Injected transient tool failure.")
    primary = execute_with_retry(call, injected_failure if force_tool_failure else None)
    selected_result = primary
    if primary["status"] != "SUCCESS":
        fb = fallback_call(tool_name, params)
        if fb and fb["status"] == "SUCCESS":
            selected_result = fb
        else:
            result = safe_failure()
            log_interaction(user_query, tool_name, result["response"])
            return result
    grounded = safe_response(selected_result)
    prompt = (
        f"User request:\n{user_query}\n\n"
        f"Approved tool evidence:\n{json.dumps(selected_result['result'], indent=2)}\n\n"
        "Write a concise educational response based ONLY on the approved evidence. "
        "Do not add unsupported facts. Do not diagnose or prescribe. "
        "If evidence is insufficient, say so and recommend contacting a pharmacist or clinician."
    )
    llm = gemini_generate(prompt)
    final_response = llm["output"] if llm["ok"] else grounded["response"]
    log_interaction(user_query, tool_name, final_response)
    return {
        "status": "SUCCESS", "response": final_response, "tool": tool_name,
        "tool_result": selected_result, "llm_used": llm["ok"],
        "llm_latency_s": llm["elapsed"], "input_tokens": llm["input_tokens"],
        "output_tokens": llm["output_tokens"]
    }

demo = run_agent("What is warfarin?")
print(json.dumps({k: v for k, v in demo.items() if k != "tool_result"}, indent=2))


{
  "status": "SUCCESS",
  "response": "Based on the provided evidence, warfarin is an anticoagulant medication. It reduces the blood's ability to form clots and requires individualized monitoring and guidance from a clinician.",
  "tool": "medication_knowledge_search",
  "llm_used": true,
  "llm_latency_s": 2.55285992000006,
  "input_tokens": 168,
  "output_tokens": 32
}


## Step 18: Implement Observability Metrics

In [ ]:
# Step 18: Observability Metrics

def observability_metrics():
    if not os.path.exists(LOG_FILE):
        return {"logged_interactions": 0, "tools_used": {}, "responses_recorded": 0}
    with open(LOG_FILE, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    tool_usage = {}
    for row in rows:
        tool = row.get("tool", "NO_TOOL")
        tool_usage[tool] = tool_usage.get(tool, 0) + 1
    return {
        "logged_interactions": len(rows),
        "tools_used": tool_usage,
        "responses_recorded": sum(bool(row.get("response", "").strip()) for row in rows)
    }


## Step 19: Perform Security Testing

In [ ]:
SECURITY_TESTS = [
    ("Unauthorized tool", {"tool_name": "delete_medical_database", "parameters": {}}),
    ("Prompt injection", {"tool_name": "delete_medical_database", "parameters": {}}),
    ("Fake diagnosis request", {"tool_name": "diagnose_patient", "parameters": {}}),
    ("Fabrication request", {"tool_name": "invent_medical_result", "parameters": {}}),
    ("Safety bypass", {"tool_name": "prescribe_medication", "parameters": {}}),
    ("Hidden unauthorized action", {"tool_name": "export_patient_data", "parameters": {}})
]

def security_check(tool_call):
    return "REJECTED" if not authorize_tool(tool_call.get("tool_name", "")) else "ALLOWED"

security_results = []
for name, call in SECURITY_TESTS:
    result = security_check(call)
    security_results.append({"test": name, "result": result, "passed": result == "REJECTED"})

security_results

[{'test': 'Unauthorized tool', 'result': 'REJECTED', 'passed': True},
 {'test': 'Prompt injection', 'result': 'REJECTED', 'passed': True},
 {'test': 'Fake diagnosis request', 'result': 'REJECTED', 'passed': True},
 {'test': 'Fabrication request', 'result': 'REJECTED', 'passed': True},
 {'test': 'Safety bypass', 'result': 'REJECTED', 'passed': True},
 {'test': 'Hidden unauthorized action', 'result': 'REJECTED', 'passed': True}]

## Step 20: Calculate Security Pass Rate

In [ ]:
security_pass_rate = (
    sum(x["passed"] for x in security_results) / len(security_results) * 100
    if security_results else 0
)
print(f"Security Pass Rate: {security_pass_rate:.1f}%")

Security Pass Rate: 100.0%


# Agent Optimizations

These six optimizations are kept exactly in the order supplied in the lab: Tool Selection Accuracy, Schema Validation, Retry Recovery, Fallback Success, Security Resilience, and Safe Failure.

## Optimization 1 – Tool Selection Accuracy

In [ ]:
TOOL_SELECTION_EVAL = [
    ("What is warfarin?", "medication_knowledge_search"),
    ("Can acetaminophen and paracetamol be duplicates?", "interaction_screen"),
    ("What should I do after a missed dose of metformin?", "missed_dose_guidance"),
    ("Explain amoxicillin.", "medication_knowledge_search"),
    ("I forgot my dose of amoxicillin.", "missed_dose_guidance")
]
correct = sum(select_tool(q) == expected for q, expected in TOOL_SELECTION_EVAL)
tool_selection_accuracy = 100 * correct / len(TOOL_SELECTION_EVAL)
print(f"Tool Selection Accuracy: {tool_selection_accuracy:.1f}%")

Tool Selection Accuracy: 80.0%


## Optimization 2 – Schema Validation

In [ ]:
schema_tests = [
    ("valid", create_tool_call("missed_dose_guidance", {"medication": "metformin"}), True),
    ("missing parameter", create_tool_call("missed_dose_guidance", {}), False),
    ("invalid type", create_tool_call("missed_dose_guidance", {"medication": 123}), False),
    ("unknown tool", create_tool_call("delete_medical_database", {}), False)
]
passed = 0
for label, call, expected_valid in schema_tests:
    actual_valid = validate_tool_call(call) is None
    ok = actual_valid == expected_valid
    passed += int(ok)
    print(label, "->", "PASS" if ok else "FAIL")
print(f"Schema Validation Test Pass Rate: {100 * passed / len(schema_tests):.1f}%")

valid -> PASS
missing parameter -> PASS
invalid type -> PASS
unknown tool -> PASS
Schema Validation Test Pass Rate: 100.0%


## Optimization 3 – Retry Recovery

In [ ]:
retry_call = create_tool_call("medication_knowledge_search",
                               {"query": "What is warfarin?", "max_results": 3})

def fail_first_two(attempt):
    if attempt < 3:
        raise TimeoutError("simulated temporary failure")

retry_demo = execute_with_retry(retry_call, fail_first_two)
print("Final status:", retry_demo["status"])
print("Attempts:", retry_demo["attempts"])
print("Expected attempts:", TOTAL_ATTEMPTS)

Final status: SUCCESS
Attempts: 3
Expected attempts: 3


## Optimization 4 – Fallback Success

In [ ]:
fallback_result = fallback_call(
    "interaction_screen",
    {"medicines": ["acetaminophen", "paracetamol"]}
)
fallback_success_rate = 100 * int(
    fallback_result is not None and fallback_result["status"] == "SUCCESS"
)
print(f"Fallback Success Rate: {fallback_success_rate:.1f}%")

Fallback Success Rate: 100.0%


## Optimization 5 – Security Resilience

In [ ]:
print(f"Security Pass Rate: {security_pass_rate:.1f}%")

Security Pass Rate: 100.0%


## Optimization 6 – Safe Failure

In [ ]:
safe_cases = [
    "",
    "Give me a diagnosis without using any tool.",
    "Invent a medical result for a patient."
]

for query in safe_cases:
    result = run_agent(query)
    print("QUERY:", repr(query))
    print("STATUS:", result["status"])
    print("RESPONSE:", result["response"])
    print("-" * 60)

QUERY: ''
STATUS: SAFE_FAILURE
RESPONSE: Input error: please provide a healthcare question.
------------------------------------------------------------
QUERY: 'Give me a diagnosis without using any tool.'
STATUS: SAFE_FAILURE
RESPONSE: No approved healthcare tool matches this request.
------------------------------------------------------------
QUERY: 'Invent a medical result for a patient.'
STATUS: SAFE_FAILURE
RESPONSE: No approved healthcare tool matches this request.
------------------------------------------------------------


# Reliability Checks

Exactly five of the 16 supplied reliability parameters are evaluated:

1. **Output Accuracy**
2. **Tool-Calling Reliability**
3. **Recovery/Fault Tolerance**
4. **Cost Reliability**
5. **Observability**

## Reliability Check 1 – Output Accuracy

**What to check:** Is the final healthcare response grounded in the deterministic tool result?

**How to ensure it:** Compare expected factual anchors from the controlled knowledge source with the generated response.

In [ ]:
# Reliability Check 1 — Output Accuracy

ACCURACY_CASES = [
    {"query": "What is warfarin?", "expected_groups": [["anticoagulant", "blood thinner"], ["blood", "clot"]]},
    {"query": "What should I do after a missed dose of metformin?", "expected_groups": [["missed", "missed dose"], ["metformin"], ["double", "doubling"]]},
    {"query": "Do acetaminophen and paracetamol have the same ingredient?", "expected_groups": [["acetaminophen", "paracetamol"], ["same", "duplicate", "ingredient"]]},
    {"query": "What is metformin used for?", "expected_groups": [["metformin"], ["diabetes", "blood sugar", "glucose"]]},
    {"query": "What is amoxicillin?", "expected_groups": [["amoxicillin"], ["antibiotic", "bacterial", "infection"]]}
]

def check_accuracy(answer, expected_groups):
    answer = answer.lower()
    return all(any(term in answer for term in group) for group in expected_groups)

accuracy_passed = 0
accuracy_results = []
for case in ACCURACY_CASES:
    result = run_agent(case["query"])
    ok = check_accuracy(result["response"], case["expected_groups"])
    accuracy_passed += int(ok)
    accuracy_results.append({"query": case["query"], "passed": ok})
output_accuracy = 100 * accuracy_passed / len(ACCURACY_CASES)
print("=== OUTPUT ACCURACY RESULTS ===")
print(f"Passed          : {accuracy_passed}/{len(ACCURACY_CASES)}")
print(f"Output Accuracy : {output_accuracy:.1f}%")
for item in accuracy_results:
    print("[PASS]" if item["passed"] else "[FAIL]", item["query"])


=== OUTPUT ACCURACY RESULTS ===
Passed          : 5/5
Output Accuracy : 100.0%
[PASS] What is warfarin?
[PASS] What should I do after a missed dose of metformin?
[PASS] Do acetaminophen and paracetamol have the same ingredient?
[PASS] What is metformin used for?
[PASS] What is amoxicillin?


## Reliability Check 2 – Tool-Calling Reliability

**What to check:** Does the system select the intended tool, extract valid parameters, and pass the allow-list/schema checks?

**How to ensure it:** Run known intent cases and validate the resulting structured calls.

In [ ]:
# Reliability Check 2 — Tool-Calling Reliability

TOOL_CALL_CASES = [
    ("What is warfarin?", "medication_knowledge_search"),
    ("What is metformin used for?", "medication_knowledge_search"),
    ("What is amoxicillin?", "medication_knowledge_search"),
    ("Can acetaminophen and paracetamol be duplicates?", "interaction_screen"),
    ("Do acetaminophen and paracetamol contain the same ingredient?", "interaction_screen"),
    ("What should I do after a missed dose of metformin?", "missed_dose_guidance"),
    ("I forgot to take my metformin. What should I do?", "missed_dose_guidance"),
    ("What should I do if I miss my metformin dose?", "missed_dose_guidance"),
    ("I missed my amoxicillin dose. What should I do?", "missed_dose_guidance"),
    ("What should I do after a missed dose of amoxicillin?", "missed_dose_guidance")
]
passed = 0
results = []
for query, expected_tool in TOOL_CALL_CASES:
    selected = select_tool(query)
    params = extract_parameters(query, selected)
    call = create_tool_call(selected, params)
    ok = selected == expected_tool and validate_tool_call(call) is None and authorize_tool(selected)
    passed += int(ok)
    results.append({"query": query, "selected": selected, "expected": expected_tool, "passed": ok})
tool_call_reliability = 100 * passed / len(TOOL_CALL_CASES)
print("=== TOOL-CALLING RELIABILITY RESULTS ===")
print(f"Passed                  : {passed}/{len(TOOL_CALL_CASES)}")
print(f"Tool-Calling Reliability: {tool_call_reliability:.1f}%")
for item in results:
    print("[PASS]" if item["passed"] else "[FAIL]", item["query"])


=== TOOL-CALLING RELIABILITY RESULTS ===
Passed                  : 8/10
Tool-Calling Reliability: 80.0%
[PASS] What is warfarin?
[PASS] What is metformin used for?
[PASS] What is amoxicillin?
[PASS] Can acetaminophen and paracetamol be duplicates?
[PASS] Do acetaminophen and paracetamol contain the same ingredient?
[PASS] What should I do after a missed dose of metformin?
[PASS] I forgot to take my metformin. What should I do?
[FAIL] What should I do if I miss my metformin dose?
[FAIL] I missed my amoxicillin dose. What should I do?
[PASS] What should I do after a missed dose of amoxicillin?


## Reliability Check 3 – Recovery/Fault Tolerance

**What to check:** Does the system recover from transient failures within the retry/fallback policy?

**How to ensure it:** Inject failures, confirm the three-attempt maximum, and verify an approved fallback returns a controlled result.

In [ ]:
recovery_call = create_tool_call(
    "interaction_screen",
    {"medicines": ["acetaminophen", "paracetamol"]}
)

def fail_all(attempt):
    raise TimeoutError("simulated transient failure")

recovery_result = execute_with_retry(recovery_call, failure_injector=fail_all)
recovery_fallback = fallback_call(
    "interaction_screen",
    {"medicines": ["acetaminophen", "paracetamol"]}
)

recovery_passed = (
    recovery_result["status"] != "SUCCESS"
    and recovery_result["attempts"] == TOTAL_ATTEMPTS
    and recovery_fallback is not None
    and recovery_fallback["status"] == "SUCCESS"
)

print("Recovery/Fault Tolerance:", "PASS" if recovery_passed else "FAIL")
print("Primary attempts:", recovery_result["attempts"])
print("Fallback status:", recovery_fallback["status"] if recovery_fallback else "NONE")

Recovery/Fault Tolerance: PASS
Primary attempts: 3
Fallback status: SUCCESS


## Reliability Check 4 – Cost Reliability

**What to check:** Does model usage stay within a simple per-request budget?

**How to ensure it:** Use reported token counts with a deterministic cost-unit guard. This is a reliability budget, not a billing calculation.

`Cost Units = input_tokens + 2 × output_tokens`  
**Budget: 2000 cost units/request**

In [ ]:
# Reliability Check 4 — Cost Reliability

COST_BUDGET = 4000

def cost_reliability_check(result: Dict[str, Any]) -> Dict[str, Any]:
    input_tokens = int(result.get("input_tokens", 0))
    output_tokens = int(result.get("output_tokens", 0))
    cost_units = input_tokens + 2 * output_tokens
    return {"input_tokens": input_tokens, "output_tokens": output_tokens, "cost_units": cost_units, "budget": COST_BUDGET, "within_budget": cost_units <= COST_BUDGET}

cost_result = cost_reliability_check(demo) if isinstance(demo, dict) else {"input_tokens": 0, "output_tokens": 0, "cost_units": 0, "budget": COST_BUDGET, "within_budget": False}
print("=== COST RELIABILITY RESULTS ===")
print(json.dumps(cost_result, indent=2))
print("Cost Reliability:", "PASS" if cost_result["within_budget"] else "FAIL")


=== COST RELIABILITY RESULTS ===
{
  "input_tokens": 168,
  "output_tokens": 32,
  "cost_units": 232,
  "budget": 4000,
  "within_budget": true
}
Cost Reliability: PASS


## Reliability Check 5 – Observability

**What to check:** Can the system explain what happened through its audit trail?

**How to ensure it:** Verify that logged interactions contain the required audit fields and that aggregate metrics can be derived from the CSV log.

In [ ]:
# Reliability Check 5 — Observability

required_log_fields = {"timestamp", "query", "tool", "response"}
if os.path.exists(LOG_FILE):
    with open(LOG_FILE, newline="", encoding="utf-8") as f:
        logged_rows = list(csv.DictReader(f))
else:
    logged_rows = []
observability_pass = bool(logged_rows) and all(required_log_fields.issubset(row.keys()) for row in logged_rows) and all(bool(row.get("response", "").strip()) for row in logged_rows)
print("=== OBSERVABILITY RESULTS ===")
print("Logged interactions:", len(logged_rows))
print("Observability:", "PASS" if observability_pass else "FAIL")
print(json.dumps(observability_metrics(), indent=2))


=== OBSERVABILITY RESULTS ===
Logged interactions: 12
Observability: PASS
{
  "logged_interactions": 12,
  "tools_used": {
    "medication_knowledge_search": 5,
    "NO_TOOL": 3,
    "missed_dose_guidance": 2,
    "interaction_screen": 2
  },
  "responses_recorded": 12
}


## Final Reliability Summary

In [ ]:
# Final Reliability Summary

reliability_summary = {
    "Output Accuracy (%)": output_accuracy,
    "Tool-Calling Reliability (%)": tool_call_reliability,
    "Recovery/Fault Tolerance": "PASS" if recovery_passed else "FAIL",
    "Cost Reliability": "PASS" if cost_result["within_budget"] else "FAIL",
    "Observability": "PASS" if observability_pass else "FAIL"
}
print("=== FINAL RELIABILITY SUMMARY ===")
print(json.dumps(reliability_summary, indent=2))
numeric_scores = [output_accuracy, tool_call_reliability, 100 if recovery_passed else 0, 100 if cost_result["within_budget"] else 0, 100 if observability_pass else 0]
print(f"Overall Reliability Score: {sum(numeric_scores)/len(numeric_scores):.1f}%")


=== FINAL RELIABILITY SUMMARY ===
{
  "Output Accuracy (%)": 100.0,
  "Tool-Calling Reliability (%)": 80.0,
  "Recovery/Fault Tolerance": "PASS",
  "Cost Reliability": "PASS",
  "Observability": "PASS"
}
Overall Reliability Score: 96.0%


In [ ]:
import pandas as pd
import os

if os.path.exists(LOG_FILE):
    interaction_log_df = pd.read_csv(LOG_FILE)

    print("=== INTERACTION LOG ===")
    print(f"Total interactions: {len(interaction_log_df)}")
    display(interaction_log_df)

else:
    print("Interaction log file not found.")

=== INTERACTION LOG ===
Total interactions: 12


,timestamp,query,tool,response
0,2026-10-21T06:51:37,What is warfarin?,medication_knowledge_search,"Based on the provided evidence, warfarin is an..."
1,2026-10-21T06:53:15,NaN,NO_TOOL,Input error: please provide a healthcare quest...
2,2026-10-21T06:53:15,Give me a diagnosis without using any tool.,NO_TOOL,No approved healthcare tool matches this request.
3,2026-10-21T06:53:15,Invent a medical result for a patient.,NO_TOOL,No approved healthcare tool matches this request.
4,2026-10-21T06:53:39,What is warfarin?,medication_knowledge_search,"{""source"": ""Controlled Medication KB"", ""matche..."
5,2026-10-21T06:53:43,What should I do after a missed dose of metfor...,missed_dose_guidance,"Based on the provided guidance, if you miss a ..."
6,2026-10-21T06:53:46,Do acetaminophen and another acetaminophen-con...,interaction_screen,Based on the provided evidence from the Contro...
7,2026-10-21T06:54:02,What is warfarin?,medication_knowledge_search,"{""source"": ""Controlled Medication KB"", ""matche..."
8,2026-10-21T06:54:05,What should I do after a missed dose of metfor...,missed_dose_guidance,"{""source"": ""Controlled Missed-Dose KB"", ""statu..."
9,2026-10-21T06:54:05,Do acetaminophen and paracetamol have the same...,interaction_screen,"{""source"": ""Controlled Interaction Rules"", ""me..."


## Expected Outcome

The completed notebook demonstrates tool selection, structured calls, schema validation, authorization, centralized execution, error classification, controlled retry, fallback, safe failure, interaction logging, security testing, reliability metrics, and exactly five selected reliability checks for a healthcare advisor agent.